# Lesson 07 - Патърн за проектиране на планиране

Този тетраден файл демонстрира **Патърна за проектиране на планиране** за AI агенти с помощта на Microsoft Agent Framework.
Ще научите как да разделяте сложни заявки за пътувания на структурираните подзадачи, да ги възлагате на специализирани агенти
и да изпълнявате получения план — всичко това с помощта на структуриран изход, захранван от Pydantic модели.


## Настройка


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Разлагане на задачата

Разлагането на задачата е същността на модела за планиране. Вместо да се иска от един агент да се справи с комплексна заявка от край до край, ние разделяме проблема на по-малки, добре дефинирани **подзадачи**. Всяка подзадача се възлага на специализиран агент (напр. полети, хотели, активности) с ясни приоритети и подредба по зависимост.

Този подход предоставя няколко предимства:
- **Яснота**: всяка подзадача има една единствена отговорност.
- **Паралелизъм**: независими подзадачи могат да се изпълняват едновременно.
- **Надеждност**: грешките са изолирани до отделни подзадачи.
- **Проследяване на бюджета**: разходите се оценяват за всяка подзадача и се събират.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Създаване на агент за планиране със структурирано изходно съдържание

Агентът за планиране действа като **координатор на рецепцията**. При получаване на общо пътешественическо запитване той
произвежда структурирано `TravelPlan` — разгражда запитването на подзадачи, определя приоритети
и идентифицира зависимости, така че консиерж или изпълнителният слой да могат да осъществят задачата.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Изпълнение на план със специализирани инструменти

След като агентът на рецепцията е съставил структуриран план, **агентът-консиерж** го изпълнява.
Всеки специализиран инструмент отговаря за една категория подзадачи (полети, хотели, дейности). Консиержът
преглежда подзадачите на плана в зависимост от реда им на зависимост и изпраща всяка една към
подходящия инструмент.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Обобщение

В този урок научихте **Патерна за планиране** за AI агенти:

1. **Декомпозиция на задачата** — Агент за планиране на рецепция разбива сложна заявка за пътуване на структурирани подзадачи с помощта на Pydantic модели, като възлага всяка на специализиран агент с приоритети и зависимости.  
2. **Структуриран изход** — Чрез подаване на `response_format` агентът връща валидиран обект `TravelPlan` вместо свободен текст, което прави по-нататъшната обработка надеждна.  
3. **Изпълнение на плана** — Агент-консиерж преминава през подзадачите, използвайки специализирани инструменти (`book_flight`, `reserve_hotel`, `book_activity`), за да изпълни плана и да докладва резултатите.  

Този патерн разделя *какво да се прави* (планиране) от *как да се прави* (изпълнение), което прави агентите по-модулни, лесни за тестване и по-лесни за разширяване.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводаческия сервис [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля, имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
